[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/02_%E5%A3%B2%E4%B8%8A%E3%83%88%E3%83%AC%E3%83%B3%E3%83%89%E3%81%AE%E5%88%86%E6%9E%90.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-02. 売上トレンドの分析（移動平均・成長率・コレログラム）

> Colab（ブラウザ）で実行可。最初に「① セットアップ」セルを実行。
> **統計検定2級レベル**（実務でよく使う検定・分析）。scipy・statsmodels を使用。

**舞台設定**：36か月の月次売上から、ならしたトレンド・成長率・季節性を読み取ります。

**使う統計（2級）**：移動平均・成長率・自己相関（コレログラム）。

### 使うデータ：`monthly_kpi.csv`（36か月の月次KPI）
`売上`(万円)の時系列を分析します。

## この章でできるようになること
- **移動平均**でノイズをならしトレンドを抽出できる
- **成長率**（前月比・前年比）で変化の勢いを測れる
- **コレログラム**（自己相関）で季節性を見抜ける

> **前提**：統計2級（時系列データの処理）　/　**所要**：約25分　/　**レベル**：2級

## 1. まず売上の推移を見る

In [ ]:
kpi = load('monthly_kpi.csv')
kpi['売上'].plot(marker='o', figsize=(10,4), title='月別売上（36か月）')
plt.ylabel('売上(万円)'); plt.show()

In [ ]:
# ここに書いて試してみよう

## 2. 移動平均でならす（トレンド抽出）

前後数か月の平均をとると、季節やノイズが消えて**トレンド**が見えます。12か月移動平均は1年の季節をならします。

In [ ]:
kpi['移動平均12'] = kpi['売上'].rolling(12).mean()
kpi[['売上','移動平均12']].plot(figsize=(10,4)); plt.ylabel('売上'); plt.show()
print('移動平均は最初の11か月分が空欄になる（前後のデータが足りないため）')

In [ ]:
# ここに書いて試してみよう

## 3. 成長率（前月比・前年比）

**成長率**は「どれだけ伸びたか」の割合。前月比は短期の勢い、前年比（12か月前との比）は季節の影響を除いた伸びを見ます。

In [ ]:
kpi['前月比%'] = kpi['売上'].pct_change() * 100
kpi['前年比%'] = kpi['売上'].pct_change(12) * 100
print('前月比の平均 %.2f%% / 前年比の平均 %.2f%%' % (kpi['前月比%'].mean(), kpi['前年比%'].mean()))
kpi[['前月比%','前年比%']].plot(figsize=(10,4)); plt.axhline(0, color='k', lw=.6); plt.ylabel('%'); plt.show()

In [ ]:
# ここに書いて試してみよう

### 平均成長率は幾何平均で求める

複数期間をならした「平均的な成長率」は、**算術平均ではなく幾何平均**で求めます。成長は毎期かけ算で効くため、算術平均だと過大になります（統計2級）。

In [ ]:
from scipy.stats import gmean
ratio = 1 + kpi['売上'].pct_change().dropna()        # 各月の (1+成長率)
geo = (gmean(ratio) - 1) * 100
ari = (ratio.mean() - 1) * 100
print(f'平均成長率: 幾何平均 {geo:.2f}%  /  算術平均 {ari:.2f}%')
print('→ 複利的に効く成長率は幾何平均が正しい')

In [ ]:
# ここに書いて試してみよう

## 4. コレログラム（自己相関）で季節性を見抜く

**自己相関**は「何か月前の自分」とどれだけ似ているか。それを並べた図が**コレログラム**です。ラグ12（12か月前）で高く出れば、1年周期の季節性があります。

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
fig, ax = plt.subplots(figsize=(8,4))
plot_acf(kpi['売上'], lags=24, ax=ax); ax.set_title('コレログラム（売上の自己相関）'); plt.show()
print('点線の帯の外に出た棒が「無相関とは言えない」自己相関。ラグ12付近に注目')

In [ ]:
# ここに書いて試してみよう

## まとめ
- **移動平均**はノイズ・季節をならしてトレンドを見せる（端は欠ける）。
- **成長率**は前月比（短期）と前年比（季節除去）で読み分ける。
- **コレログラム**はラグ12の山で季節性を示す。

> **よくある間違い**
> - 移動平均は窓の幅だけ端のデータが欠ける（最初の11か月など）。
> - 前月比だけ見ると季節変動に振り回される。季節性があるときは**前年比**も見る。
> - 自己相関が高い＝周期がある、だが「なぜ」は別途要因分析が必要。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 売上が 100→120 になったときの成長率(%) を ans に
ans = None   # 例: (120-100)/100*100
_check('Q1 成長率%', ans, (120-100)/100*100)

In [ ]:
# Q2: [10,20,30] の3項移動平均（最後の値）を ans に
ans = None   # 例: (10+20+30)/3
_check('Q2 移動平均', ans, (10+20+30)/3)

In [ ]:
# Q3: 季節性が1年周期のとき、コレログラムで自己相関が高く出るラグ(月)を ans に
ans = None   # ヒント: 12か月周期
_check('Q3 季節のラグ', ans, 12)

---
## 練習問題

**問1.** `受注数` で同じ分析（移動平均・前年比・コレログラム）をしよう。季節性はある？

**問2.** 3か月移動平均と12か月移動平均を重ねて描き、ならし方の違いを観察しよう。

**問3.**（考察）前月比が大きく上下するのに前年比が安定している場合、何が起きているか説明しよう。

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/02_%E5%A3%B2%E4%B8%8A%E3%83%88%E3%83%AC%E3%83%B3%E3%83%89%E3%81%AE%E5%88%86%E6%9E%90.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 移動平均 | 前後の平均でならした系列 |
| 成長率 | 期間あたりの増減の割合 |
| 自己相関 | 過去の自分との相関 |
| コレログラム | 自己相関をラグ順に並べた図 |

```python
s.rolling(12).mean()          # 12か月移動平均
s.pct_change(12)*100          # 前年比%
from statsmodels.graphics.tsaplots import plot_acf; plot_acf(s, lags=24)
```

## 完了ログ

このノートを終えたら下のセルを実行すると、学習の完了が記録されます。
**学習者コードは最初に一度だけ設定**すれば、以降は全ノートで自動送信されます（名前の再入力は不要）。

- Colab：左の鍵アイコン（シークレット）で `STUDENT_CODE` に配布コードを登録（1回だけ）
- ローカル：環境変数 `STUDENT_CODE` を設定（または初回に画面入力すると保存され、次回から不要）

In [ ]:
# === 完了ログ ===  学習者コードは最初に一度だけ設定すれば、全ノートで自動送信されます。
import os, datetime, pathlib
try:
    import requests
except ImportError:
    requests = None

def _student_code():
    try:                                          # Colab: 鍵アイコンで STUDENT_CODE を登録
        from google.colab import userdata
        c = userdata.get('STUDENT_CODE')
        if c: return c.strip()
    except Exception:
        pass
    c = os.environ.get('STUDENT_CODE')            # ローカル: 環境変数
    if c: return c.strip()
    p = pathlib.Path.home() / '.student_code'      # 保存ファイル
    if p.exists(): return p.read_text().strip()
    try:                                           # 最後の手段: 入力して保存（次回から不要）
        c = input('学習者コードを入力（配布されたもの）: ').strip()
        if c: p.write_text(c); return c
    except Exception:
        pass
    return ''

CODE = _student_code()
LOG_URL = ""      # 配布時に設定
LOG_TOKEN = ""    # 配布時に設定
NOTEBOOK = "06_発展_マーケ分析/02_売上トレンドの分析"

if CODE and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "code": CODE, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {CODE} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
elif not CODE:
    print("学習者コード未設定。Colabは鍵アイコンで STUDENT_CODE を登録、ローカルは環境変数 STUDENT_CODE を設定してください。")
else:
    print(f"{NOTEBOOK}: LOG_URL/LOG_TOKEN が未設定です（配布時に設定されます）。")